# 아발론(Abalone) AlphaZero 학습 노트북

육각형 판에서 구슬을 밀어내는 **아발론**을 **ResNet + MCTS 자기대전**으로 학습합니다.

**학습 방식 (AlphaGo Zero 스타일)**
1. **자기대전**: 현재 *최고 모델*끼리 MCTS로 대국하며 데이터를 모읍니다.
2. **학습**: *학습 모델*이 버퍼의 데이터로 정책(어떤 수가 좋은가)과 가치(누가 이길까)를 배웁니다.
3. **아레나**: 학습 모델 vs 최고 모델을 대국시켜 **55% 이상 이기면 최고 모델을 교체**합니다.
4. 반복마다 구글 드라이브에 저장하므로, 세션이 끊겨도 다시 **위에서부터 실행하면 이어서 학습**합니다.

**v2 개선점**
- **대칭 데이터 증강**: 육각판의 12가지 회전·반사로 학습 데이터를 늘려, 같은 대국 수로 더 많이 배웁니다.
- **가치 목표 개선**: 승패만이 아니라 떨어뜨린 구슬 차이도 가치 목표에 반영해, "구슬을 떨어뜨리고 지키는 것"을 더 빨리 배웁니다.
- 신경망 구조는 그대로라 **v1에서 학습하던 체크포인트를 그대로 이어서** 쓸 수 있습니다.

**v3 개선점** (반복 75회 전후, 정책손실 정체 대응)
- **계단식 학습률**: `lr_schedule`에 따라 반복 0~74는 1e-3, 75~149는 3e-4, 150부터는 1e-4. 이어서 학습할 때도 설정한 학습률이 적용됩니다.
- **탐색 횟수 64 → 128**: 자기대전 정답지(MCTS 결과)의 품질을 높입니다. 반복 시간은 약 2배로 늘어납니다.
- **평가 판 수 10 → 20**: 휴리스틱 상대 승률이 덜 출렁이도록.

**v4 개선점** (반복 86회 전후, 무승부 증가·소극적 플레이 대응)
- **휴리스틱 대국 섞기**: 자기대전 중 `heuristic_frac`(기본 25%)만큼은 휴리스틱 봇을 상대로 두고, AI 쪽 국면만 학습 데이터로 씁니다. 공격적인 상대를 직접 겪으면서 수비만 하는 플레이를 깨뜨리고, 휴리스틱의 전략에 대응하는 법을 배웁니다.
- 출력에 **자기대전 vs휴리스틱** 성적이 매 반복 함께 나옵니다(반복마다 16판).
- 참고: "비기면 양쪽 다 감점"하는 무승부 페널티는 MCTS의 부호 뒤집기 구조 때문에 오히려 비기는 쪽으로 유도해서 넣지 않았습니다.

**v5 개선점** (반복 105회 전후, 휴리스틱 연습판 0% 대응)
- 휴리스틱과 두는 판에서는 AI가 **초반 4수만 무작위**(`heuristic_temp_moves`)로 두고 **탐색 잡음을 끕니다**(`heuristic_noise=False`). 초반을 대충 둬서 시작부터 지는 판 대신, 제대로 싸워서 이기거나 지는 판으로 배웁니다.
- 순수 자기대전 판은 그대로(초반 30수 무작위 + 탐색 잡음)라 다양성은 유지됩니다.

**v6 개선점** (반복 109회 전후, 평가 전용 상대 추가)
- AI가 v4부터 휴리스틱 봇과 직접 두며 배우기 때문에, 휴리스틱 승률만으로는 "진짜 실력"인지 "그 봇에 맞춘 실력"인지 구분이 어렵습니다.
- 그래서 학습에는 쓰지 않는 **평가 전용 미니맥스 봇**(2수 앞까지 읽음, 휴리스틱을 약 96% 이김)을 추가했습니다. 3번마다 20판씩 평가합니다.
- 휴리스틱 승률과 미니맥스 승률이 같이 오르면 진짜 실력이 는 것, 휴리스틱만 오르면 그 봇에 맞춰진 것입니다.

**시작 전에**: 메뉴 `런타임 → 런타임 유형 변경 → GPU(T4)`를 선택하세요.

**규칙 메모**: 한 번에 1~3개 구슬을 직선 또는 옆걸음으로 이동, 수가 더 많으면 상대 구슬을 밀 수 있음(3:2, 3:1, 2:1).
상대 구슬 6개를 먼저 떨어뜨리면 승리. 수비만 하다 끝나지 않는 걸 막으려고 `max_plies`(기본 200수)에 도달하면
떨어뜨린 구슬이 많은 쪽이 이기고, 같으면 무승부입니다.

## 1. 설정 & 드라이브 연결
설정값을 바꾸고 싶으면 `CFG`를 수정하세요. T4 기준 반복 1회에 대략 10~20분 걸립니다.

In [ ]:
# ==== 설정 & 구글 드라이브 연결 ====
import os, torch

try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/abalone_az'   # 체크포인트 저장 위치
except ImportError:
    SAVE_DIR = './abalone_az'                        # Colab이 아닐 때
os.makedirs(SAVE_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('장치:', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '')
print('저장 위치:', SAVE_DIR)

CFG = dict(
    # 신경망
    channels=64,            # ResNet 채널 수
    blocks=6,               # 잔차 블록 수
    # 게임
    max_plies=200,          # 최대 수(넘으면 떨어뜨린 구슬 수로 판정)
    # 자기대전
    games_per_iter=64,      # 반복마다 자기대전 판 수
    parallel_games=32,      # 동시에 진행할 판 수 (GPU 배치 크기)
    sims=128,               # 한 수당 MCTS 시뮬레이션 횟수 (v3: 64 → 128)
    c_puct=1.5,
    dirichlet_alpha=0.3,
    dirichlet_eps=0.25,
    temp_moves=30,          # 처음 몇 수는 방문 비율대로 무작위 선택
    heuristic_frac=0.25,    # v4: 자기대전 중 휴리스틱 봇과 두는 판의 비율
    heuristic_temp_moves=4, # v5: 휴리스틱 판에서 AI가 무작위로 두는 초반 수 (자기대전은 temp_moves)
    heuristic_noise=False,  # v5: 휴리스틱 판에서 탐색 잡음(디리클레) 사용 여부
    # 학습
    buffer_size=150_000,
    batch_size=256,
    train_steps=400,        # 반복마다 학습 스텝 수
    lr_schedule={0: 1e-3, 75: 3e-4, 150: 1e-4},   # 반복 번호: 학습률 (계단식으로 낮춤)
    weight_decay=1e-4,
    augment=True,           # 육각판 12가지 대칭으로 데이터 증강
    result_weight=0.5,      # 가치 목표 = 0.5×승패 + 0.5×(떨어뜨린 구슬 차이/6)
    # 아레나(새 모델 vs 현재 최고 모델)
    arena_games=20,
    arena_sims=64,
    arena_temp_moves=4,     # 아레나 판이 매번 같아지지 않도록 초반 몇 수만 무작위
    accept_rate=0.55,       # 이 이상 이기면 최고 모델 교체
    # 평가(랜덤·휴리스틱 상대)
    eval_every=3,
    eval_games=20,
    # 전체
    iterations=200,         # 100회로는 부족해서 200회로 늘림
)


## 2. 게임 환경
61칸 육각판, 축 좌표 사용. 행동은 직선 이동 366개 + 옆걸음 2196개 = 2562개로 인코딩합니다.

In [ ]:
# ==== 아발론(Abalone) 게임 환경 ====
# 61칸 육각판을 축 좌표 (q, r)로 표현합니다. |q|, |r|, |q+r| <= 4
# 흑 = 1, 백 = -1, 빈칸 = 0. 흑이 먼저 둡니다.
import math, random, time, os, json, pickle, copy
from collections import deque
import numpy as np

CELLS = [(q, r) for r in range(-4, 5) for q in range(-4, 5)
         if abs(q) <= 4 and abs(r) <= 4 and abs(q + r) <= 4]
NCELL = len(CELLS)                       # 61
CELL_INDEX = {c: i for i, c in enumerate(CELLS)}
# 6방향. 반대 방향 = (d + 3) % 6
DIRS = [(1, 0), (1, -1), (0, -1), (-1, 0), (-1, 1), (0, 1)]
DIR_NAMES = ['E', 'NE', 'NW', 'W', 'SW', 'SE']

NEIGH = [[CELL_INDEX.get((q + dq, r + dr), -1) for (dq, dr) in DIRS] for (q, r) in CELLS]
GRID_R = np.array([r + 4 for (q, r) in CELLS])   # 신경망 입력용 9x9 격자 위치
GRID_C = np.array([q + 4 for (q, r) in CELLS])
CENTER_DIST = [max(abs(q), abs(r), abs(q + r)) for (q, r) in CELLS]

# ---- 행동 인코딩 ----
# 직선 이동(inline): 꼬리 칸 c에서 방향 d로 줄 전체를 민다       -> c*6 + d
# 옆걸음(broadside): c에서 축 ax(0~2) 방향으로 L(2~3)개, 방향 d로 이동
#                    -> 366 + ((c*3 + ax)*2 + (L-2))*6 + d
N_INLINE = NCELL * 6
ACTION_SIZE = N_INLINE + NCELL * 3 * 2 * 6      # 2562


def cell_name(i):
    """표준 표기: 줄 A(아래)~I(위), 열 1~9"""
    q, r = CELLS[i]
    return 'ABCDEFGHI'[4 - r] + str(q + 5)


class State:
    __slots__ = ('board', 'player', 'lost', 'ply')

    def __init__(self, board, player, lost, ply):
        self.board = board      # 길이 61 리스트
        self.player = player    # 둘 차례 (1 또는 -1)
        self.lost = lost        # [흑이 잃은 구슬 수, 백이 잃은 구슬 수]
        self.ply = ply          # 지금까지 둔 수

    def copy(self):
        return State(list(self.board), self.player, list(self.lost), self.ply)


def lost_idx(color):
    return 0 if color == 1 else 1


def initial_state():
    board = [0] * NCELL
    for i, (q, r) in enumerate(CELLS):
        if r in (3, 4) or (r == 2 and -2 <= q <= 0):
            board[i] = 1
        if r in (-3, -4) or (r == -2 and 0 <= q <= 2):
            board[i] = -1
    return State(board, 1, [0, 0], 0)


def legal_actions(s):
    b, me = s.board, s.player
    opp = -me
    acts = []
    for c in range(NCELL):
        if b[c] != me:
            continue
        nc = NEIGH[c]
        # 직선 이동 (1개짜리 이동 포함)
        for d in range(6):
            n, x = 0, c
            while x != -1 and b[x] == me and n <= 3:
                n += 1
                x = NEIGH[x][d]
            if n > 3 or x == -1:          # 4개 이상이거나 내 구슬이 판 밖으로
                continue
            if b[x] == 0:
                acts.append(c * 6 + d)
                continue
            m, y = 0, x                    # 상대 구슬 밀기(스모)
            while y != -1 and b[y] == opp:
                m += 1
                y = NEIGH[y][d]
            if m < n and (y == -1 or b[y] == 0):
                acts.append(c * 6 + d)
        # 옆걸음
        for ax in range(3):
            c2 = nc[ax]
            if c2 == -1 or b[c2] != me:
                continue
            c3 = NEIGH[c2][ax]
            groups = [(2, (c, c2))]
            if c3 != -1 and b[c3] == me:
                groups.append((3, (c, c2, c3)))
            for L, grp in groups:
                base = N_INLINE + ((c * 3 + ax) * 2 + (L - 2)) * 6
                for d in range(6):
                    if d == ax or d == ax + 3:
                        continue
                    ok = True
                    for g in grp:
                        t = NEIGH[g][d]
                        if t == -1 or b[t] != 0:
                            ok = False
                            break
                    if ok:
                        acts.append(base + d)
    return acts


def apply_action(s, a):
    """새 State를 반환 (원래 State는 그대로)"""
    ns = s.copy()
    b, me = ns.board, ns.player
    if a < N_INLINE:
        c, d = divmod(a, 6)
        x = c
        while b[x] == me:
            x = NEIGH[x][d]
        b[c] = 0
        if b[x] == -me:                    # 밀어내기
            y = x
            while y != -1 and b[y] == -me:
                y = NEIGH[y][d]
            if y == -1:
                ns.lost[lost_idx(-me)] += 1  # 판 밖으로 떨어뜨림
            else:
                b[y] = -me
        b[x] = me
    else:
        k, d = divmod(a - N_INLINE, 6)
        k, L2 = divmod(k, 2)
        c, ax = divmod(k, 3)
        grp = [c]
        for _ in range(L2 + 1):
            grp.append(NEIGH[grp[-1]][ax])
        for g in grp:
            b[g] = 0
        for g in grp:
            b[NEIGH[g][d]] = me
    ns.player = -me
    ns.ply += 1
    return ns


def winner(s, max_plies):
    """승자(1/-1), 무승부 0, 진행 중이면 None.
    최대 수에 도달하면 더 많이 떨어뜨린 쪽이 승리(같으면 무승부)."""
    if s.lost[0] >= 6:
        return -1
    if s.lost[1] >= 6:
        return 1
    if s.ply >= max_plies:
        diff = s.lost[1] - s.lost[0]
        return (diff > 0) - (diff < 0)
    return None


def outcome_value(s, w, player, result_weight=0.5):
    """끝난 판의 가치(player 관점). 승패만이 아니라 떨어뜨린 구슬 차이도 반영해서
    '구슬을 많이 떨어뜨릴수록 좋다'는 신호를 더 촘촘하게 줍니다."""
    margin = (s.lost[lost_idx(-player)] - s.lost[lost_idx(player)]) / 6.0
    return float(np.clip(result_weight * w * player + (1 - result_weight) * margin, -1, 1))


# ---- 육각판 대칭 12가지 (회전 6 × 반사 2) : 데이터 증강용 ----
def _sym_point(k, q, r):
    x, z = q, r
    y = -x - z
    if k >= 6:                      # 반사
        y, z = z, y
    for _ in range(k % 6):          # 60도 회전
        x, y, z = -z, -x, -y
    return x, z


SYM_CELL = np.array([[CELL_INDEX[_sym_point(k, q, r)] for (q, r) in CELLS] for k in range(12)])
SYM_DIR = np.array([[DIRS.index(_sym_point(k, dq, dr)) for (dq, dr) in DIRS] for k in range(12)])


def _build_sym_actions():
    tab = np.full((12, ACTION_SIZE), -1, np.int64)
    for k in range(12):
        for c in range(NCELL):
            for d in range(6):
                tab[k, c * 6 + d] = SYM_CELL[k, c] * 6 + SYM_DIR[k, d]
            for ax in range(3):
                for L in (2, 3):
                    grp = [c]
                    for _ in range(L - 1):
                        nxt = NEIGH[grp[-1]][ax]
                        if nxt == -1:
                            break
                        grp.append(nxt)
                    if len(grp) < L:
                        continue
                    tax = SYM_DIR[k, ax]
                    if tax < 3:
                        start, nax = SYM_CELL[k, grp[0]], tax
                    else:
                        start, nax = SYM_CELL[k, grp[-1]], tax - 3
                    for d in range(6):
                        if d == ax or d == ax + 3:
                            continue
                        src = N_INLINE + ((c * 3 + ax) * 2 + (L - 2)) * 6 + d
                        tab[k, src] = N_INLINE + ((start * 3 + nax) * 2 + (L - 2)) * 6 + SYM_DIR[k, d]
    return tab


SYM_ACTION = _build_sym_actions()


def action_name(s, a):
    if a < N_INLINE:
        c, d = divmod(a, 6)
        x, n = c, 0
        while s.board[x] == s.player:
            n += 1
            x = NEIGH[x][d]
        push = ' (밀기)' if s.board[x] == -s.player else ''
        return f'{cell_name(c)} 직선 {n}개 → {DIR_NAMES[d]}{push}'
    k, d = divmod(a - N_INLINE, 6)
    k, L2 = divmod(k, 2)
    c, ax = divmod(k, 3)
    return f'{cell_name(c)}~{DIR_NAMES[ax]} {L2 + 2}개 옆걸음 → {DIR_NAMES[d]}'


def render(s):
    sym = {1: '●', -1: '○', 0: '·'}
    lines = []
    for r in range(-4, 5):
        row = [sym[s.board[CELL_INDEX[(q, r)]]] for q in range(-4, 5) if (q, r) in CELL_INDEX]
        lines.append(' ' * abs(r) + 'ABCDEFGHI'[4 - r] + ' ' + ' '.join(row))
    lines.append(f'흑● 잃은 구슬 {s.lost[0]} | 백○ 잃은 구슬 {s.lost[1]} | {s.ply}수 | 차례: {"흑●" if s.player == 1 else "백○"}')
    return '\n'.join(lines)


def encode(s, max_plies):
    """신경망 입력: (6, 9, 9) — 항상 '둘 차례인 사람' 관점"""
    x = np.zeros((6, 9, 9), np.float32)
    b = np.array(s.board, np.int8) * s.player
    x[0, GRID_R, GRID_C] = (b == 1)
    x[1, GRID_R, GRID_C] = (b == -1)
    x[2, GRID_R, GRID_C] = 1.0
    x[3] = s.lost[lost_idx(s.player)] / 6.0
    x[4] = s.lost[lost_idx(-s.player)] / 6.0
    x[5] = s.ply / max_plies
    return x


### 규칙 확인
시작 국면의 합법수는 44개여야 합니다(아발론의 알려진 값).

In [ ]:
s = initial_state()
print(render(s))
print('시작 합법수:', len(legal_actions(s)))

# 랜덤 대국 몇 판으로 구슬 개수 보존 확인
for _ in range(5):
    s = initial_state()
    while winner(s, 400) is None:
        s = apply_action(s, random.choice(legal_actions(s)))
        assert sum(v == 1 for v in s.board) + s.lost[0] == 14
        assert sum(v == -1 for v in s.board) + s.lost[1] == 14
print('규칙 테스트 통과')


## 3. 비교용 상대 (랜덤 / 휴리스틱 / 미니맥스)
휴리스틱은 한 수 앞만 보고, 구슬 떨어뜨리기 > 가운데 차지 > 뭉쳐 있기 순으로 점수를 매깁니다.
미니맥스(평가 전용)는 같은 점수표로 상대의 응수까지 2수 앞을 읽습니다.

In [ ]:
# ==== 비교용 상대: 랜덤 / 휴리스틱 ====
class RandomAgent:
    name = 'random'

    def select_moves(self, states):
        return [random.choice(legal_actions(s)) for s in states]


def heuristic_score(s, me):
    """me 관점 점수: 떨어뜨린 구슬 >> 가운데 차지 > 뭉쳐 있기"""
    score = 1000 * (s.lost[lost_idx(-me)] - s.lost[lost_idx(me)])
    b = s.board
    for i in range(NCELL):
        v = b[i]
        if v == 0:
            continue
        sign = 1 if v == me else -1
        score -= sign * 3 * CENTER_DIST[i]
        for d in range(3):
            j = NEIGH[i][d]
            if j != -1 and b[j] == v:
                score += sign * 1
    return score


class HeuristicAgent:
    """한 수 앞만 보는 탐욕 에이전트"""
    name = 'heuristic'

    def select_moves(self, states):
        out = []
        for s in states:
            best, best_a = -1e9, None
            for a in legal_actions(s):
                sc = heuristic_score(apply_action(s, a), s.player) + random.random()
                if sc > best:
                    best, best_a = sc, a
            out.append(best_a)
        return out


class MinimaxAgent:
    """평가 전용: 2수 앞(내 수 → 상대의 최선 응수)까지 읽는 미니맥스 봇.
    점수표는 휴리스틱과 같지만, 상대가 어떻게 받아칠지까지 따져서 함정을 피합니다.
    학습 데이터에는 쓰지 않고 실력 측정용으로만 씁니다."""
    name = 'minimax'

    def select_moves(self, states):
        return [self._best(s) for s in states]

    def _best(self, s):
        me = s.player
        # 좋아 보이는 수부터 보면 가지치기가 잘 돼서 빨라짐
        cand = []
        for a in legal_actions(s):
            s1 = apply_action(s, a)
            cand.append((heuristic_score(s1, me) + random.random(), a, s1))
        cand.sort(key=lambda t: -t[0])
        best, best_a = -1e18, cand[0][1]
        for _, a, s1 in cand:
            w = winner(s1, CFG['max_plies'])
            if w is not None:
                v = 1e9 * w * me
            else:
                v, cut = 1e18, False
                for b in legal_actions(s1):
                    # 같은 점수일 때 무작위로 고르도록 아주 작은 값을 더함
                    sc = heuristic_score(apply_action(s1, b), me) + random.random() * 0.5
                    if sc < v:
                        v = sc
                        if v <= best:          # 이미 더 좋은 수가 있으면 나머지 응수는 볼 필요 없음
                            cut = True
                            break
                if cut:                        # 끝까지 안 본 수는 후보에서 제외
                    continue
            if v > best:
                best, best_a = v, a
        return best_a


## 4. ResNet (정책 + 가치)

In [ ]:
# ==== ResNet (정책 + 가치) ====
import torch.nn as nn
import torch.nn.functional as F


class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.c1 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.b1 = nn.BatchNorm2d(ch)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.b2 = nn.BatchNorm2d(ch)

    def forward(self, x):
        y = F.relu(self.b1(self.c1(x)))
        y = self.b2(self.c2(y))
        return F.relu(x + y)


class AbaloneNet(nn.Module):
    def __init__(self, channels=64, blocks=6):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(6, channels, 3, padding=1, bias=False),
                                  nn.BatchNorm2d(channels), nn.ReLU())
        self.body = nn.Sequential(*[ResBlock(channels) for _ in range(blocks)])
        self.p_head = nn.Sequential(nn.Conv2d(channels, 8, 1, bias=False), nn.BatchNorm2d(8), nn.ReLU(),
                                    nn.Flatten(), nn.Linear(8 * 81, ACTION_SIZE))
        self.v_head = nn.Sequential(nn.Conv2d(channels, 2, 1, bias=False), nn.BatchNorm2d(2), nn.ReLU(),
                                    nn.Flatten(), nn.Linear(2 * 81, 128), nn.ReLU(),
                                    nn.Linear(128, 1), nn.Tanh())

    def forward(self, x):
        h = self.body(self.stem(x))
        return self.p_head(h), self.v_head(h).squeeze(1)


def new_net():
    return AbaloneNet(CFG['channels'], CFG['blocks']).to(DEVICE)


@torch.no_grad()
def predict(net, states):
    """여러 국면을 한 번에 GPU로 평가 -> (정책 로짓, 가치)"""
    net.eval()
    x = torch.from_numpy(np.stack([encode(s, CFG['max_plies']) for s in states])).to(DEVICE)
    with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
        logits, v = net(x)
    return logits.float().cpu().numpy(), v.float().cpu().numpy()


## 5. MCTS
여러 판의 탐색을 동시에 진행하고, 잎 국면들을 모아 **한 번에 GPU로 평가**해서 GPU를 효율적으로 씁니다.

In [ ]:
# ==== MCTS (여러 판의 탐색을 묶어서 GPU 배치 평가) ====
class Node:
    __slots__ = ('state', 'actions', 'P', 'N', 'W', 'children', 'terminal', 'value')

    def __init__(self, state):
        self.state = state
        self.actions = None
        self.children = {}
        w = winner(state, CFG['max_plies'])
        self.terminal = w is not None
        # 가치는 항상 '이 국면에서 둘 차례인 사람' 관점
        self.value = None if w is None else outcome_value(state, w, state.player, CFG['result_weight'])

    def expand(self, logits):
        acts = legal_actions(self.state)
        if not acts:                       # 둘 수가 없으면 패배
            self.terminal, self.value = True, -1.0
            return
        self.actions = np.array(acts)
        lg = logits[self.actions]
        p = np.exp(lg - lg.max())
        self.P = p / p.sum()
        self.N = np.zeros(len(acts), np.float32)
        self.W = np.zeros(len(acts), np.float32)

    def select(self, c_puct):
        Q = np.divide(self.W, self.N, out=np.zeros_like(self.W), where=self.N > 0)
        U = c_puct * self.P * math.sqrt(self.N.sum() + 1) / (1 + self.N)
        return int(np.argmax(Q + U))


def backup(path, v):
    # v: 잎 국면에서 둘 차례인 사람 관점의 가치. 한 칸 올라갈 때마다 부호가 바뀜
    for node, i in reversed(path):
        v = -v
        node.N[i] += 1
        node.W[i] += v


def run_mcts(net, states, sims, add_noise=False):
    # add_noise: True/False 또는 국면마다 따로 정하는 리스트
    noise_flags = add_noise if isinstance(add_noise, (list, tuple)) else [add_noise] * len(states)
    roots = [Node(s) for s in states]
    logits, _ = predict(net, states)
    for r, lg, nz in zip(roots, logits, noise_flags):
        r.expand(lg)
        if nz and r.actions is not None:
            noise = np.random.dirichlet([CFG['dirichlet_alpha']] * len(r.actions))
            r.P = (1 - CFG['dirichlet_eps']) * r.P + CFG['dirichlet_eps'] * noise
    c = CFG['c_puct']
    for _ in range(sims):
        leaves, paths = [], []
        for r in roots:
            if r.terminal:
                continue
            node, path = r, []
            while not node.terminal and node.actions is not None:
                i = node.select(c)
                path.append((node, i))
                child = node.children.get(i)
                if child is None:
                    child = Node(apply_action(node.state, int(node.actions[i])))
                    node.children[i] = child
                node = child
            if node.terminal:
                backup(path, node.value)
            else:
                leaves.append(node)
                paths.append(path)
        if leaves:
            logits, values = predict(net, [n.state for n in leaves])
            for n, path, lg, v in zip(leaves, paths, logits, values):
                n.expand(lg)
                backup(path, n.value if n.terminal else float(v))
    return roots


def pick_action(root, temperature):
    if temperature == 0:
        return int(root.actions[int(np.argmax(root.N))])
    p = root.N / root.N.sum()
    return int(np.random.choice(root.actions, p=p))


class MCTSAgent:
    def __init__(self, net, sims, temp_moves=0, name='mcts'):
        self.net, self.sims, self.temp_moves, self.name = net, sims, temp_moves, name

    def select_moves(self, states):
        roots = run_mcts(self.net, states, self.sims)
        return [pick_action(r, 1.0 if s.ply < self.temp_moves else 0) for r, s in zip(roots, states)]


## 6. 자기대전 / 학습 / 아레나

In [ ]:
# ==== 자기대전 / 학습 / 아레나 ====
def self_play(net, n_games):
    """최고 모델로 대국하며 (국면, MCTS 방문 분포, 최종 결과) 샘플을 모읍니다.
    v4: 전체 판 중 heuristic_frac 비율은 휴리스틱 봇을 상대로 두고, AI 쪽 국면만 학습 데이터로 씁니다."""
    samples, lengths, decisive = [], [], 0
    heur, h_res = HeuristicAgent(), []
    n_h = int(round(n_games * CFG['heuristic_frac']))
    # 판마다 휴리스틱이 맡을 색 (None = 순수 자기대전). 흑백을 번갈아 맡김
    kinds = [None] * (n_games - n_h) + [1 if i % 2 == 0 else -1 for i in range(n_h)]
    random.shuffle(kinds)
    done = 0
    while done < n_games:
        batch = min(CFG['parallel_games'], n_games - done)
        hcol = kinds[done:done + batch]
        states = [initial_state() for _ in range(batch)]
        hist = [[] for _ in range(batch)]
        active = list(range(batch))
        while active:
            h_turn = [g for g in active if hcol[g] == states[g].player]
            m_turn = [g for g in active if hcol[g] != states[g].player]
            if h_turn:                                   # 휴리스틱 차례
                for g, a in zip(h_turn, heur.select_moves([states[g] for g in h_turn])):
                    states[g] = apply_action(states[g], a)
            if m_turn:                                   # AI(MCTS) 차례
                # v5: 휴리스틱 판에서는 실전처럼(잡음 없이) 탐색
                noise = [hcol[g] is None or CFG['heuristic_noise'] for g in m_turn]
                roots = run_mcts(net, [states[g] for g in m_turn], CFG['sims'], add_noise=noise)
                for g, root in zip(m_turn, roots):
                    s = states[g]
                    pi = root.N / root.N.sum()
                    # 저장 공간을 아끼기 위해 국면을 압축해서 보관
                    hist[g].append((np.array(s.board, np.int8), s.player, s.lost[0], s.lost[1], s.ply,
                                    root.actions.astype(np.int16), pi.astype(np.float16)))
                    tm = CFG['temp_moves'] if hcol[g] is None else CFG['heuristic_temp_moves']
                    temp = 1.0 if s.ply < tm else 0
                    states[g] = apply_action(s, pick_action(root, temp))
            still = []
            for g in active:
                w = winner(states[g], CFG['max_plies'])
                if w is None:
                    still.append(g)
                    continue
                for rec in hist[g]:
                    z = outcome_value(states[g], w, rec[1], CFG['result_weight'])
                    samples.append(rec + (z,))
                lengths.append(states[g].ply)
                decisive += (w != 0)
                if hcol[g] is not None:
                    h_res.append(-w * hcol[g])           # AI 관점 결과 (1 승, 0 무, -1 패)
            active = still
        done += batch
    stats = dict(avg_len=float(np.mean(lengths)), decisive=decisive / n_games)
    if h_res:
        stats['sp_vs_heuristic'] = (sum(r == 1 for r in h_res) + 0.5 * sum(r == 0 for r in h_res)) / len(h_res)
    return samples, stats


def rec_to_state(rec, sym=0):
    board = rec[0]
    if sym:
        nb = np.zeros(NCELL, np.int8)
        nb[SYM_CELL[sym]] = board          # 판을 회전/반사
        board = nb
    return State(list(board), int(rec[1]), [int(rec[2]), int(rec[3])], int(rec[4]))


def train(net, opt, buffer):
    net.train()
    tot_p = tot_v = 0.0
    for _ in range(CFG['train_steps']):
        batch = random.sample(buffer, min(CFG['batch_size'], len(buffer)))
        # 데이터 증강: 샘플마다 12가지 대칭 중 하나를 무작위로 적용
        syms = np.random.randint(12, size=len(batch)) if CFG['augment'] else np.zeros(len(batch), int)
        x = np.stack([encode(rec_to_state(r, k), CFG['max_plies']) for r, k in zip(batch, syms)])
        pi = np.zeros((len(batch), ACTION_SIZE), np.float32)
        mask = np.full((len(batch), ACTION_SIZE), -1e4, np.float32)
        for j, (r, k) in enumerate(zip(batch, syms)):
            acts = SYM_ACTION[k][r[5]]         # 행동도 같은 대칭으로 변환
            pi[j, acts] = r[6]
            mask[j, acts] = 0
        z = np.array([r[7] for r in batch], np.float32)
        x, pi, mask, z = (torch.from_numpy(a).to(DEVICE) for a in (x, pi, mask, z))
        logits, v = net(x)
        loss_p = -(pi * F.log_softmax(logits + mask, dim=1)).sum(1).mean()
        loss_v = F.mse_loss(v, z)
        opt.zero_grad()
        (loss_p + loss_v).backward()
        opt.step()
        tot_p += loss_p.item()
        tot_v += loss_v.item()
    n = CFG['train_steps']
    return tot_p / n, tot_v / n


def play_match(agent_a, agent_b, n_games):
    """a 관점 성적 (승, 무, 패). 흑/백을 번갈아 맡습니다."""
    states = [initial_state() for _ in range(n_games)]
    a_color = [1 if g % 2 == 0 else -1 for g in range(n_games)]
    result = [None] * n_games
    active = list(range(n_games))
    while active:
        for agent, is_a in ((agent_a, True), (agent_b, False)):
            group = [g for g in active if (states[g].player == a_color[g]) == is_a]
            if group:
                moves = agent.select_moves([states[g] for g in group])
                for g, m in zip(group, moves):
                    states[g] = apply_action(states[g], m)
            still = []
            for g in active:
                w = winner(states[g], CFG['max_plies'])
                if w is None:
                    still.append(g)
                else:
                    result[g] = w * a_color[g]
            active = still
    wins = sum(r == 1 for r in result)
    draws = sum(r == 0 for r in result)
    return wins, draws, n_games - wins - draws


## 7. 메인 학습 루프
이 셀을 실행하면 학습이 시작됩니다. 중간에 멈춰도(■ 버튼) 마지막으로 끝난 반복까지 저장되어 있어요.

In [ ]:
# ==== 메인 학습 루프 (중간에 끊겨도 드라이브에서 이어서 학습) ====
CKPT = os.path.join(SAVE_DIR, 'checkpoint.pt')
BUF = os.path.join(SAVE_DIR, 'buffer.pkl')
LOG = os.path.join(SAVE_DIR, 'log.json')

trainee = new_net()       # 계속 학습하는 모델
best = new_net()          # 자기대전에 쓰는 현재 최고 모델


def lr_at(it):
    """lr_schedule에서 현재 반복에 해당하는 학습률"""
    return CFG['lr_schedule'][max(k for k in CFG['lr_schedule'] if k <= it)]


opt = torch.optim.Adam(trainee.parameters(), lr=lr_at(0), weight_decay=CFG['weight_decay'])
buffer = deque(maxlen=CFG['buffer_size'])
log, start_it = [], 0

if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=DEVICE)
    trainee.load_state_dict(ck['trainee'])
    best.load_state_dict(ck['best'])
    opt.load_state_dict(ck['opt'])
    start_it = ck['iteration'] + 1
    if os.path.exists(BUF):
        with open(BUF, 'rb') as f:
            buffer.extend(pickle.load(f))
    if os.path.exists(LOG):
        log = json.load(open(LOG))
    print(f'{start_it}번째 반복부터 이어서 학습 (버퍼 {len(buffer)}개)')
else:
    best.load_state_dict(trainee.state_dict())
    print('처음부터 학습 시작')


def save(it):
    tmp = CKPT + '.tmp'
    torch.save(dict(trainee=trainee.state_dict(), best=best.state_dict(),
                    opt=opt.state_dict(), iteration=it, cfg=CFG), tmp)
    os.replace(tmp, CKPT)
    torch.save(best.state_dict(), os.path.join(SAVE_DIR, 'best_model.pt'))
    with open(BUF + '.tmp', 'wb') as f:
        pickle.dump(list(buffer), f)
    os.replace(BUF + '.tmp', BUF)
    json.dump(log, open(LOG, 'w'), ensure_ascii=False, indent=1)


for it in range(start_it, CFG['iterations']):
    t0 = time.time()
    # 0) 학습률 적용 (저장된 옵티마이저의 예전 학습률을 설정값으로 덮어씀)
    lr = lr_at(it)
    if opt.param_groups[0]['lr'] != lr:
        print(f'학습률 {opt.param_groups[0]["lr"]:g} → {lr:g}')
    for g in opt.param_groups:
        g['lr'] = lr
    # 1) 최고 모델로 자기대전
    samples, sp = self_play(best, CFG['games_per_iter'])
    buffer.extend(samples)
    t1 = time.time()
    # 2) 학습
    lp, lv = train(trainee, opt, buffer)
    t2 = time.time()
    # 3) 아레나: 새 모델 vs 최고 모델
    w, d, l = play_match(MCTSAgent(trainee, CFG['arena_sims'], CFG['arena_temp_moves']),
                         MCTSAgent(best, CFG['arena_sims'], CFG['arena_temp_moves']),
                         CFG['arena_games'])
    score = (w + 0.5 * d) / CFG['arena_games']
    accepted = score >= CFG['accept_rate']
    if accepted:
        best.load_state_dict(trainee.state_dict())
    t3 = time.time()
    entry = dict(it=it, lr=lr, policy_loss=lp, value_loss=lv, arena=[w, d, l], arena_score=score,
                 accepted=accepted, buffer=len(buffer), **sp,
                 time=dict(selfplay=t1 - t0, train=t2 - t1, arena=t3 - t2))
    # 4) 가끔 랜덤·휴리스틱·미니맥스 상대로 최고 모델 실력 확인
    if it % CFG['eval_every'] == 0:
        agent = MCTSAgent(best, CFG['arena_sims'], CFG['arena_temp_moves'])
        for opp in (RandomAgent(), HeuristicAgent(), MinimaxAgent()):   # v6: 미니맥스는 평가 전용
            ew, ed, el = play_match(agent, opp, CFG['eval_games'])
            entry[f'vs_{opp.name}'] = (ew + 0.5 * ed) / CFG['eval_games']
    log.append(entry)
    save(it)
    print(f"[{it:3d}] 정책손실 {lp:.3f} 가치손실 {lv:.3f} | 아레나 {w}승 {d}무 {l}패 "
          f"{'✅ 교체' if accepted else '❌ 유지'} | 평균 {sp['avg_len']:.0f}수, 승부난 비율 {sp['decisive']:.0%}"
          + (f" | vs랜덤 {entry['vs_random']:.0%} vs휴리스틱 {entry['vs_heuristic']:.0%} vs미니맥스 {entry['vs_minimax']:.0%}" if 'vs_random' in entry else '')
          + (f" | 자기대전 vs휴리스틱 {sp['sp_vs_heuristic']:.0%}" if 'sp_vs_heuristic' in sp else '')
          + f" | {time.time() - t0:.0f}초")


## 8. 학습 그래프

In [ ]:
# ==== 학습 그래프 ====
import matplotlib.pyplot as plt

its = [e['it'] for e in log]
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(its, [e['policy_loss'] for e in log], label='policy loss')
ax[0].plot(its, [e['value_loss'] for e in log], label='value loss')
ax[0].set_title('Loss'); ax[0].legend()
ax[1].plot(its, [e['arena_score'] for e in log], 'o-', label='new vs best')
acc = [e['it'] for e in log if e['accepted']]
ax[1].scatter(acc, [CFG['accept_rate']] * len(acc), marker='^', color='green', label='model replaced')
ax[1].axhline(CFG['accept_rate'], ls='--', color='gray')
ax[1].set_ylim(0, 1); ax[1].set_title('Arena score'); ax[1].legend()
ev = [e for e in log if 'vs_random' in e]
ax[2].plot([e['it'] for e in ev], [e['vs_random'] for e in ev], 'o-', label='vs random')
ax[2].plot([e['it'] for e in ev], [e['vs_heuristic'] for e in ev], 'o-', label='vs heuristic')
evm = [e for e in ev if 'vs_minimax' in e]
ax[2].plot([e['it'] for e in evm], [e['vs_minimax'] for e in evm], 'o-', label='vs minimax (eval only)')
ax[2].set_ylim(0, 1); ax[2].set_title('Best model score'); ax[2].legend()
for a in ax:
    a.set_xlabel('iteration')
plt.tight_layout(); plt.show()


## 9. AI 대국 구경하기

In [ ]:
# ==== 최고 모델(흑) vs 휴리스틱(백) 한 판 구경하기 ====
from IPython.display import clear_output

def watch_game(agent_black, agent_white, delay=0.3, max_show=None):
    s = initial_state()
    while winner(s, CFG['max_plies']) is None:
        agent = agent_black if s.player == 1 else agent_white
        a = agent.select_moves([s])[0]
        desc = action_name(s, a)
        s = apply_action(s, a)
        clear_output(wait=True)
        print(f'{agent.name}: {desc}\n')
        print(render(s))
        time.sleep(delay)
    w = winner(s, CFG['max_plies'])
    print('\n결과:', {1: '흑 승', -1: '백 승', 0: '무승부'}[w])

watch_game(MCTSAgent(best, 128, name='AI'), HeuristicAgent())


## 10. 직접 AI와 두기

In [ ]:
# ==== 직접 AI와 두기 ====
# 합법수 목록에서 번호를 입력하세요. 'q'를 입력하면 종료.
def play_human(human_color=1, sims=200):
    ai = MCTSAgent(best, sims, name='AI')
    s = initial_state()
    while winner(s, CFG['max_plies']) is None:
        print(render(s))
        if s.player == human_color:
            acts = legal_actions(s)
            for k, a in enumerate(acts):
                print(f'{k:3d}: {action_name(s, a)}')
            cmd = input('번호 입력: ').strip()
            if cmd == 'q':
                return
            if not cmd.isdigit() or int(cmd) >= len(acts):
                print('잘못된 번호예요.\n')
                continue
            s = apply_action(s, acts[int(cmd)])
        else:
            a = ai.select_moves([s])[0]
            print('AI:', action_name(s, a), '\n')
            s = apply_action(s, a)
    print(render(s))
    w = winner(s, CFG['max_plies'])
    print('결과:', '무승부' if w == 0 else ('당신 승리!' if w == human_color else 'AI 승리'))

# play_human(human_color=1)   # 1 = 흑(먼저 둠), -1 = 백. 주석을 풀고 실행하세요.


## 11. 판을 클릭해서 AI와 두기 (추천)
번호 입력 대신 **판을 직접 클릭해서** 둡니다. Colab에서만 동작합니다.

1. 1~6번 셀을 실행합니다. 7번 학습 루프는 실행하지 않아도 됩니다. 저장된 최고 모델(`best_model.pt`)을 자동으로 불러옵니다.
2. 이 셀을 실행하면 판이 나타납니다.
3. **내 구슬을 1~3개 클릭**해서 고르면(한 줄로 붙어 있어야 함) 갈 수 있는 칸이 **초록색**으로 표시됩니다. **빨간 테두리**는 밀 수 있는 상대 구슬입니다.
4. 초록 칸이나 빨간 테두리를 클릭하면 이동합니다. 2~3개를 옆걸음으로 옮길 때는 **마지막으로 클릭한 구슬**이 갈 칸을 누르세요.
5. 빈 곳을 클릭하면 선택이 취소됩니다. `새 게임`, `무르기` 버튼도 있습니다.

AI의 세기는 마지막 줄의 `sims`로 바꿀 수 있습니다(클수록 세지만 느려짐).

In [ ]:
# ==== 판을 클릭해서 AI와 두기 (Colab 전용) ====
# 사용법: 1~6번 셀을 실행한 뒤 이 셀을 실행하세요. (7번 학습 루프는 실행하지 않아도 됩니다)
# - 내 구슬을 1~3개 클릭해 고르면(한 줄로 붙어 있어야 함) 갈 수 있는 칸이 초록색으로 표시됩니다.
# - 초록 칸을 클릭하면 이동합니다. 빨간 테두리 = 상대 구슬 밀기.
# - 2~3개를 옆걸음으로 옮길 때는 "마지막으로 클릭한 구슬"이 갈 칸을 누르세요.
import json, html
from IPython.display import HTML, JSON, display
from google.colab import output


def _move_info(s, a):
    """행동 a로 움직이는 내 구슬들, 방향, 클릭할 목표 칸(직선 이동)"""
    if a < N_INLINE:
        c, d = divmod(a, 6)
        grp, x = [c], NEIGH[c][d]
        while x != -1 and s.board[x] == s.player:
            grp.append(x)
            x = NEIGH[x][d]
        return grp, d, x
    k, d = divmod(a - N_INLINE, 6)
    k, L2 = divmod(k, 2)
    c, ax = divmod(k, 3)
    grp = [c]
    for _ in range(L2 + 1):
        grp.append(NEIGH[grp[-1]][ax])
    return grp, d, None


def _is_line(cells):
    """1~3개가 한 줄로 붙어 있는지"""
    if len(cells) == 1:
        return True
    if len(cells) > 3:
        return False
    cs = set(cells)
    for a in cells:
        for d in range(6):
            run, x = [a], NEIGH[a][d]
            while x != -1 and x in cs:
                run.append(x)
                x = NEIGH[x][d]
            if len(run) == len(cells):
                return True
    return False


class ClickGame:
    def __init__(self, net, human_color=1, sims=400):
        self.ai = MCTSAgent(net, sims, name='AI')
        self.new(human_color)

    def new(self, human_color):
        self.human = human_color
        self.s = initial_state()
        self.hist, self.log, self.sel, self.anchor, self.last = [], [], [], None, []

    # ---- 목표 칸 계산 ----
    def _targets(self):
        if not self.sel or self.s.player != self.human or self._over():
            return {}
        out, S = {}, set(self.sel)
        for a in legal_actions(self.s):
            grp, d, front = _move_info(self.s, a)
            if set(grp) != S:
                continue
            if front is not None:                       # 직선 이동: 앞쪽 칸(밀 상대 구슬 포함)
                key = front
            else:                                       # 옆걸음: 마지막으로 고른 구슬이 갈 칸
                key = NEIGH[self.anchor if self.anchor in S else grp[0]][d]
            out[key] = a
        return out

    def _over(self):
        return winner(self.s, CFG['max_plies']) is not None

    def _apply(self, a, who):
        before = list(self.s.board)
        self.log.append(f'{who}: {action_name(self.s, a)}')
        self.s = apply_action(self.s, a)
        self.last = [i for i in range(NCELL) if before[i] != self.s.board[i]]
        self.sel, self.anchor = [], None

    # ---- 화면에 보낼 상태 ----
    def payload(self, msg=''):
        tg = self._targets()
        w = winner(self.s, CFG['max_plies'])
        over = None
        if w is not None:
            over = '무승부' if w == 0 else ('당신 승리! 🎉' if w == self.human else 'AI 승리')
        return dict(board=self.s.board, lost=self.s.lost, ply=self.s.ply, maxp=CFG['max_plies'],
                    player=self.s.player, human=self.human, sel=self.sel, last=self.last,
                    targets=[dict(c=c, push=self.s.board[c] == -self.human, label=action_name(self.s, a))
                             for c, a in tg.items()],
                    log=self.log[-8:], over=over, msg=msg, can_undo=bool(self.hist))

    # ---- 클릭 처리 ----
    def click(self, i):
        i = int(i)
        if self._over() or self.s.player != self.human:
            return self.payload()
        tg = self._targets()
        if i in tg:                                     # 목표 칸 클릭 → 이동
            self.hist.append((self.s.copy(), list(self.log), list(self.last)))
            self._apply(tg[i], '나')
            return self.payload()
        if self.s.board[i] == self.human:               # 내 구슬 클릭 → 고르기
            if i in self.sel:
                rest = [c for c in self.sel if c != i]
                self.sel = rest if rest and _is_line(rest) else []
                self.anchor = self.sel[-1] if self.sel else None
            else:
                cand = self.sel + [i]
                self.sel = cand if _is_line(cand) else [i]
                self.anchor = i
            msg = '' if self._targets() or not self.sel else '이 조합으로는 갈 수 있는 칸이 없어요'
            return self.payload(msg)
        self.sel, self.anchor = [], None                # 빈 곳 클릭 → 선택 취소
        return self.payload()

    def ai_move(self):
        if not self._over() and self.s.player != self.human:
            a = self.ai.select_moves([self.s])[0]
            self._apply(a, 'AI')
        return self.payload()

    def undo(self):
        if self.hist:
            self.s, self.log, self.last = self.hist.pop()
            self.sel, self.anchor = [], None
        return self.payload()


_HTML = r"""
<div id="abl" style="font-family:sans-serif;user-select:none">
  <div style="margin-bottom:6px">
    <button id="abl-newb">새 게임 (내가 흑●, 먼저)</button>
    <button id="abl-neww">새 게임 (내가 백○)</button>
    <button id="abl-undo">무르기</button>
  </div>
  <div id="abl-status" style="font-size:15px;margin:4px 0;font-weight:bold"></div>
  <div style="display:flex;gap:16px;align-items:flex-start;flex-wrap:wrap">
    <svg id="abl-svg" width="500" height="460"></svg>
    <div style="min-width:240px">
      <div id="abl-info" style="font-size:14px;margin-bottom:6px"></div>
      <div id="abl-hint" style="font-size:13px;color:#555;margin-bottom:6px"></div>
      <div id="abl-log" style="font-size:13px;line-height:1.5;color:#333;white-space:pre"></div>
    </div>
  </div>
</div>
<script>
(function(){
  const CELLS = __CELLS__;
  const S = 46, CX = 250, CY = 228;
  const pos = CELLS.map(([q,r]) => [CX + (q + r/2)*S, CY + r*S*0.866]);
  const svg = document.getElementById('abl-svg');
  let st = __INIT__, busy = false;

  async function call(name, args){
    try {
      const res = await google.colab.kernel.invokeFunction(name, args || [], {});
      const d = res && res.data && res.data['application/json'];
      if (d && d.board) return d;
      throw new Error('응답이 비어 있어요');
    } catch (e) {
      st.msg = '오류: ' + (e.message || e) + ' — 이 셀을 다시 실행해 주세요';
      return st;
    }
  }
  function xy(q, r){ return [CX + (q + r/2)*S, CY + r*S*0.866]; }

  function draw(){
    const tmap = {}; st.targets.forEach(t => tmap[t.c] = t);
    let h = '';
    const corners = [[-4,0],[0,-4],[4,-4],[4,0],[0,4],[-4,4]].map(([q,r]) => {
      const [x,y] = xy(q,r); return [CX + (x-CX)*1.13, CY + (y-CY)*1.13];
    });
    h += `<polygon points="${corners.map(p=>p.join(',')).join(' ')}" fill="#b5835a" stroke="#7a5230" stroke-width="3"/>`;
    for (let r = -4; r <= 4; r++){            // 줄 이름 A~I
      const qmin = Math.max(-4, -4 - r); const [x,y] = xy(qmin, r);
      h += `<text x="${x - S*0.78}" y="${y+5}" font-size="13" fill="#3b2a1a" text-anchor="middle">${'ABCDEFGHI'[4-r]}</text>`;
    }
    for (let q = -4; q <= 4; q++){            // 열 번호 1~9
      const rmax = Math.min(4, 4 - q); const [x,y] = xy(q, rmax + 0.72);
      h += `<text x="${x}" y="${y+5}" font-size="13" fill="#3b2a1a" text-anchor="middle">${q+5}</text>`;
    }
    CELLS.forEach((_, i) => {
      const [x,y] = pos[i], v = st.board[i], t = tmap[i];
      h += `<circle cx="${x}" cy="${y}" r="19" fill="#8a5d3b" opacity="0.55"/>`;
      if (st.last.includes(i)) h += `<circle cx="${x}" cy="${y}" r="22" fill="none" stroke="#ffd54f" stroke-width="2" stroke-dasharray="4 3"/>`;
      if (v !== 0){
        const fill = v === 1 ? '#1e1e1e' : '#f5f5f0', stroke = v === 1 ? '#000' : '#999';
        h += `<circle cx="${x}" cy="${y}" r="18" fill="${fill}" stroke="${stroke}" stroke-width="1.5"/>`;
        h += `<circle cx="${x-5}" cy="${y-6}" r="5" fill="#fff" opacity="${v===1?0.18:0.6}"/>`;
      }
      if (st.sel.includes(i)) h += `<circle cx="${x}" cy="${y}" r="21" fill="none" stroke="#ffeb3b" stroke-width="4"/>`;
      if (t){
        if (t.push) h += `<circle cx="${x}" cy="${y}" r="21" fill="none" stroke="#e53935" stroke-width="4"/>`;
        else h += `<circle cx="${x}" cy="${y}" r="12" fill="#43a047" opacity="0.85"/>`;
      }
      h += `<circle data-i="${i}" cx="${x}" cy="${y}" r="22" fill="transparent" style="cursor:pointer"><title>${t ? t.label : ''}</title></circle>`;
    });
    svg.innerHTML = h;

    const me = st.human === 1 ? '흑●' : '백○', ai = st.human === 1 ? '백○' : '흑●';
    let status;
    if (st.over) status = '결과: ' + st.over;
    else if (busy) status = 'AI 생각 중… (' + ai + ')';
    else status = st.player === st.human ? ('당신 차례 (' + me + ')') : ('AI 차례 (' + ai + ')');
    document.getElementById('abl-status').textContent = status;
    document.getElementById('abl-info').innerHTML =
      `흑● 잃은 구슬: <b>${st.lost[0]}</b> / 6<br>백○ 잃은 구슬: <b>${st.lost[1]}</b> / 6<br>${st.ply} / ${st.maxp}수`;
    document.getElementById('abl-hint').textContent = st.msg ||
      (st.sel.length ? '초록 칸을 누르면 이동해요 (빨간 테두리 = 밀기)' : '움직일 내 구슬을 1~3개 클릭하세요');
    document.getElementById('abl-log').textContent = st.log.join('\n');
    document.getElementById('abl-undo').disabled = !st.can_undo || busy;
  }

  async function aiIfNeeded(){
    if (!st.over && st.player !== st.human){
      busy = true; draw();
      try { st = await call('abalone.ai'); } finally { busy = false; }
      draw();
    }
  }
  svg.addEventListener('click', async (e) => {
    const el = e.target.closest('[data-i]');
    if (!el || busy || st.over || st.player !== st.human) return;
    st = await call('abalone.click', [parseInt(el.dataset.i)]);
    draw();
    await aiIfNeeded();
  });
  document.getElementById('abl-newb').onclick = async () => { if (busy) return; st = await call('abalone.new', [1]); draw(); await aiIfNeeded(); };
  document.getElementById('abl-neww').onclick = async () => { if (busy) return; st = await call('abalone.new', [-1]); draw(); await aiIfNeeded(); };
  document.getElementById('abl-undo').onclick = async () => { if (busy) return; st = await call('abalone.undo'); draw(); };
  draw();
  aiIfNeeded();
})();
</script>
"""


def play_click(human_color=1, sims=400):
    """판을 클릭해서 AI와 둡니다. human_color: 1 = 흑(먼저 둠), -1 = 백"""
    g = globals()
    if 'best' not in g:                                 # 7번을 안 돌렸으면 저장된 최고 모델을 불러옴
        net = new_net()
        net.load_state_dict(torch.load(os.path.join(SAVE_DIR, 'best_model.pt'), map_location=DEVICE))
        g['best'] = net
    game = ClickGame(g['best'], human_color, sims)
    output.register_callback('abalone.click', lambda i: JSON(game.click(i)))
    output.register_callback('abalone.ai', lambda: JSON(game.ai_move()))
    output.register_callback('abalone.new', lambda c: JSON((game.new(int(c)), game.payload())[1]))
    output.register_callback('abalone.undo', lambda: JSON(game.undo()))
    page = _HTML.replace('__CELLS__', json.dumps(CELLS)).replace('__INIT__', json.dumps(game.payload()))
    display(HTML(page))


play_click(human_color=1, sims=400)   # 1 = 흑(먼저 둠), -1 = 백


## 더 강하게 만들고 싶다면
- `sims`, `arena_sims`를 늘리면 수 읽기가 깊어지지만 느려집니다.
- `channels`, `blocks`를 키우면 모델이 커집니다(GPU 여유가 있을 때).
- 게임 로직을 numba로 최적화하면 CPU 쪽 병목이 줄어듭니다.
- 그래도 휴리스틱 봇을 못 이기면, 자기대전에 휴리스틱 봇과의 대국을 섞어 상대의 전략을 직접 경험하게 할 수 있습니다.